# Recovering missing awake EEG motifs: a diagnostic walkthrough

This notebook investigates **where EEG motif information may be getting lost**. The scientific question is not merely whether an algorithm returns a motif. It is why the original sign-based method can find a clean pattern in some LFP recordings while failing on some awake EEG channels.

We will inspect the pipeline in this order:

1. Load one extracted EEG window and identify valid channels.
2. Inspect raw voltage before any motif processing.
3. Apply common-average referencing (CAR) and see what it removes.
4. Measure channel noise and spectral shape.
5. Convert one channel into smoothed frequency-band envelopes.
6. Compare continuous envelopes with the binary signs used by the original method.
7. Compare binary motifs, continuous hierarchical motifs, and GMM motifs.
8. Check whether the 4–8 Hz pattern follows running.
9. Optionally run the full analysis across every valid EEG channel.
10. Compare the beginning and end of an isoflurane window.

> **Important:** GMM always returns the requested number of clusters. A GMM cluster is only a *candidate motif*. Stability, null controls, behavior, and held-out data determine whether it is meaningful.

## 0. Setup and safety switches

The default input below is a small extracted awake HDF5 file included in the repository. Change `INPUT_PATH` to analyze another extracted window.

The expensive switches are initially `False`. This lets us understand one channel before running every channel. For quick debugging, `N_NULL_DEMO=3` is acceptable. Final results should use substantially more null permutations, such as 100, because three nulls are not enough for a trustworthy significance claim.

In [ ]:
from pathlib import Path
import sys

# Find the repository root whether Jupyter starts from the repo or notebook folder.
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'code/python/src').exists():
    REPO = REPO.parent
if not (REPO / 'code/python/src').exists():
    raise RuntimeError('Could not find the dc-allen repository root.')

sys.path.insert(0, str(REPO / 'code/python'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.eeg_motif_diagnostics import (
    channel_metrics,
    common_average_reference,
    load_window,
    run_diagnostics,
)
from src.get_motifs import get_motifs
from src.gmm_motifs import bic_sweep, fit_gmm_motifs, seed_stability, shift_bands
from src.smooth_env import DEFAULT_FREQ_BANDS, smooth_env

INPUT_PATH = REPO / 'data/sample_nwb/688277-20230921-v001_awake.h5'
OUTPUT_DIR = REPO / 'data/out/eeg_motif_diagnostics_notebook'

CHANNEL = None          # None chooses the first valid channel; or enter a number such as 13.
N_NULL_DEMO = 3         # Fast demonstration only; use about 100 for final results.
GMM_K = 5               # A hypothesis to inspect, not automatically the true number of motifs.
RUN_FULL_AUDIT = False  # Set True only when ready to process every good channel.
RUN_TRANSITION_CHECK = False  # Set True when INPUT_PATH is an isoflurane window.

print('Repository:', REPO)
print('Input exists:', INPUT_PATH.exists(), INPUT_PATH)

## 1. Load the extracted window

The EEG array is organized as **time samples × channels**. `is_data_valid` comes from the electrode metadata and tells us which channels the recording marked as usable. We should not include invalid channels in the common average, because a broken channel could contaminate every other channel.

This cell does not transform the signal. It only confirms what data we actually have.

In [ ]:
window = load_window(INPUT_PATH)
eeg = window['eeg']
ts = window['ts']
fs = window['fs']
valid = window['valid']
valid_channels = np.flatnonzero(valid)

print(f'EEG shape: {eeg.shape} = time samples × channels')
print(f'Sampling rate: {fs:.2f} Hz')
print(f'Duration: {(ts[-1] - ts[0]):.1f} seconds')
print(f'Valid channels: {valid.sum()} / {len(valid)}')
print('Window metadata:', window['attrs'])

channel_table = pd.DataFrame({
    'channel': np.arange(eeg.shape[1]),
    'location': window['location'],
    'valid': valid,
})
channel_table

### What to look for

- Are most channels valid, or are only a few contributing?
- Do valid and invalid channels cluster in particular anatomical locations?
- Is the window long enough to contain repeated patterns? A motif cannot be established from a single brief event.

## 2. Inspect raw EEG before preprocessing

A motif algorithm can only work with information present in its input. We first plot a short raw segment from several valid channels. Shared deflections appearing almost identically everywhere may be common reference noise, movement artifact, or a genuinely global brain signal. A pattern isolated to one channel may instead be local activity or a faulty electrode.

In [ ]:
seconds_to_show = 15
n_show = min(len(ts), int(seconds_to_show * fs))
picks = valid_channels[np.linspace(0, len(valid_channels) - 1, min(5, len(valid_channels))).round().astype(int)]
t_show = ts[:n_show] - ts[0]

fig, axes = plt.subplots(len(picks), 1, figsize=(13, 1.6 * len(picks)), sharex=True)
axes = np.atleast_1d(axes)
for ax, ch in zip(axes, picks):
    ax.plot(t_show, eeg[:n_show, ch], lw=0.6)
    ax.set_ylabel(f'ch {ch}\n{window["location"][ch]}')
    ax.spines[['top', 'right']].set_visible(False)
axes[-1].set_xlabel('Time from window start (s)')
fig.suptitle('Raw EEG: do events appear locally or across every channel?')
plt.tight_layout()
plt.show()

## 3. Common-average referencing (CAR)

At every sample, CAR averages the voltage over all valid EEG channels and subtracts that average from each valid channel:

$$x_{CAR,c}(t) = x_c(t) - \frac{1}{C}\sum_{j=1}^{C}x_j(t)$$

Why do this? If the same artifact is added to every channel, the common average estimates that shared contamination. Subtraction can expose channel-specific activity. But CAR can also remove genuine widespread neural activity, so we must compare **with and without** it rather than assuming CAR is always better.

In [ ]:
eeg_car, common_reference = common_average_reference(eeg, valid)
chosen_channel = int(valid_channels[0] if CHANNEL is None else CHANNEL)
if not valid[chosen_channel]:
    raise ValueError(f'Channel {chosen_channel} is marked invalid; choose a valid channel.')

fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
axes[0].plot(t_show, eeg[:n_show, chosen_channel], color='0.45', lw=0.7)
axes[0].set_ylabel('Original EEG')
axes[1].plot(t_show, common_reference[:n_show], color='#d95f02', lw=0.7)
axes[1].set_ylabel('Common average')
axes[2].plot(t_show, eeg_car[:n_show, chosen_channel], color='#1769aa', lw=0.7)
axes[2].set_ylabel('After CAR')
axes[2].set_xlabel('Time from window start (s)')
fig.suptitle(f'Channel {chosen_channel} ({window["location"][chosen_channel]}): what does CAR remove?')
plt.tight_layout()
plt.show()

### Interpreting CAR

- If a suspicious oscillation is large in the common average and shrinks after CAR across many channels, it may be shared contamination.
- If it remains strong only in anatomically related channels, it may be a real regional signal.
- If all meaningful structure disappears, CAR may be removing a genuine global brain state. That is why both versions remain in the comparison.

## 4. Channel-level exploratory analysis

We calculate five summaries for every channel:

- **Robust SD:** typical voltage variability, less sensitive to rare spikes than ordinary SD.
- **Peak-to-peak voltage:** catches extreme excursions and possible artifacts.
- **Theta fraction:** the fraction of 1–100 Hz power in 6–10 Hz, relevant to the suspicious 7–8 Hz line.
- **Line-noise fraction:** the fraction near 60 Hz, which can reveal electrical contamination.
- **Spectral slope:** how quickly power falls with frequency between 2 and 40 Hz.

These measurements help explain *why* some channels succeed while others fail.

In [ ]:
metrics_original = channel_metrics(eeg, fs).add_suffix('_original')
metrics_car = channel_metrics(eeg_car, fs).add_suffix('_car')
metrics = channel_table.join(metrics_original).join(metrics_car)
metrics_valid = metrics[metrics.valid].copy()
metrics_valid.sort_values('theta_fraction_original', ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, field, label in [
    (axes[0], 'robust_sd_uv', 'Robust SD'),
    (axes[1], 'theta_fraction', '6–10 Hz power fraction'),
    (axes[2], 'line_noise_fraction', '59–61 Hz power fraction'),
]:
    ax.plot(metrics_valid.channel, metrics_valid[f'{field}_original'], 'o-', ms=3, label='original')
    ax.plot(metrics_valid.channel, metrics_valid[f'{field}_car'], 'o-', ms=3, label='CAR')
    ax.set(xlabel='EEG channel', ylabel=label)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False)
fig.suptitle('Which channels are noisy, theta-heavy, or dominated by line noise?')
plt.tight_layout()
plt.show()

## 5. Turn one EEG channel into frequency-band envelopes

`smooth_env` performs the main preprocessing transformation:

1. Band-pass filter the raw EEG into the listed frequency ranges.
2. Use the Hilbert transform to estimate instantaneous amplitude in each band.
3. Smooth the amplitude for roughly five seconds.
4. Z-score each band across the window.
5. Mark ten seconds at each edge as NaN because filtering is unreliable there.

The output is **time × frequency bands**. Positive values mean a band is stronger than its own average during this window; negative values mean weaker.

In [ ]:
band_labels = [f'{lo:g}–{hi:g} Hz' for lo, hi in DEFAULT_FREQ_BANDS]
env_original = smooth_env(eeg[:, chosen_channel], fs, DEFAULT_FREQ_BANDS)
env_car = smooth_env(eeg_car[:, chosen_channel], fs, DEFAULT_FREQ_BANDS)
finite = np.isfinite(env_original).all(axis=1) & np.isfinite(env_car).all(axis=1)

print('Frequency bands:', band_labels)
print('Envelope shape:', env_original.shape, '= time samples × bands')
print('Samples remaining after removing filter edges:', finite.sum())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
extent = [ts[0] - ts[0], ts[-1] - ts[0], -0.5, len(band_labels) - 0.5]
for ax, env, title in [
    (axes[0], env_original, 'Original reference'),
    (axes[1], env_car, 'Common-average reference'),
]:
    image = ax.imshow(env.T, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-2.5, vmax=2.5, extent=extent)
    ax.set_yticks(range(len(band_labels)), band_labels)
    ax.set_title(title)
axes[-1].set_xlabel('Time in window (s)')
fig.colorbar(image, ax=axes, label='Band envelope (z-score)', shrink=0.8)
fig.suptitle(f'Channel {chosen_channel}: did CAR change the spectral patterns?')
plt.show()

## 6. The exact information lost during binarization

The original motif algorithm samples the envelopes at about one value per second and replaces every value with only its sign:

- `+1`: this band is above its window average.
- `-1`: this band is below its window average.

For example, `+0.05` and `+3.0` both become `+1`. That makes the method robust and interpretable, but it can erase amplitude differences that may be especially important for weaker scalp EEG.

In [ ]:
hop = max(1, int(round(fs)))
env_clean = env_original[finite]
time_clean = ts[finite]
env_1hz = env_clean[::hop]
time_1hz = time_clean[::hop]
signs_1hz = np.sign(env_1hz)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, values, title, limit in [
    (axes[0], env_1hz, 'Continuous 1 Hz band envelopes: magnitude is preserved', 2.5),
    (axes[1], signs_1hz, 'Binary signs: magnitude has been discarded', 1),
]:
    ax.imshow(values.T, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit,
              extent=[time_1hz[0] - ts[0], time_1hz[-1] - ts[0], -0.5, len(band_labels) - 0.5])
    ax.set_yticks(range(len(band_labels)), band_labels)
    ax.set_title(title)
axes[-1].set_xlabel('Time in window (s)')
plt.tight_layout()
plt.show()

### What would support the binarization hypothesis?

If the continuous heatmap contains repeated, graded spectral shapes but the binary heatmap collapses them into mostly identical columns, binarization is plausibly hiding useful EEG structure. If both representations look unstructured, the problem probably occurs earlier—in the raw signal, referencing, spectral decomposition, or channel selection.

## 7. Compare three definitions of a motif

We now compare:

1. **Binary hierarchical motifs:** the MATLAB-compatible method.
2. **Continuous hierarchical motifs:** the same clustering and null logic, but using envelope magnitudes instead of signs. This isolates the effect of binarization.
3. **GMM motifs:** a different model that groups seconds with similar continuous spectral profiles.

A fair comparison uses the same channel, time window, frequency bands, and preprocessing.

In [ ]:
binary_result = get_motifs(
    [env_clean.T], fs=fs, n_null=N_NULL_DEMO, seed=0, representation='binary'
)
continuous_result = get_motifs(
    [env_clean.T], fs=fs, n_null=N_NULL_DEMO, seed=0, representation='continuous'
)

print('Binary significant motif pairs:', binary_result.n_motifs)
print('Continuous significant motif pairs:', continuous_result.n_motifs)
print('Reminder: N_NULL_DEMO is intentionally small, so these are exploratory counts.')

In [ ]:
def plot_hierarchical_motifs(result, title):
    if result.n_motifs == 0:
        print(title, '— no significant motif pair found')
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    for index in range(result.motifs.shape[1]):
        pole = '+' if index < result.n_motifs else '−'
        pair = index % result.n_motifs + 1
        ax.plot(band_labels, result.motifs[:, index], marker='o', label=f'M{pair}{pole}')
    ax.axhline(0, color='0.7', lw=0.8)
    ax.set(ylabel='Mean motif value', title=title)
    ax.tick_params(axis='x', rotation=35)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_hierarchical_motifs(binary_result, 'Binary hierarchical motifs')
plot_hierarchical_motifs(continuous_result, 'Continuous hierarchical motifs')

## 8. GMM motifs, K selection, and repeatability

GMM represents every cluster as a mean vector across frequency bands. It assigns every second to one cluster with a probability. Unlike the original algorithm, it does not require opposing `+` and `−` poles and does not automatically reject clusters using the original circular-shift null.

We therefore inspect three safeguards:

- **BIC across K:** a lower value is preferred, although a continually decreasing curve means K remains unresolved.
- **Seed stability:** adjusted Rand index (ARI) near 1 means different random starts produce nearly the same grouping.
- **Shifted-band control:** shifting frequency bands relative to one another destroys their simultaneous relationship while retaining each band's individual time series.

In [ ]:
ks, bic_real = bic_sweep(env_1hz, range(1, 13), seed=0)
_, bic_shifted = bic_sweep(shift_bands(env_1hz), range(1, 13), seed=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, bic_real, 'o-', label='real simultaneous bands')
ax.plot(ks, bic_shifted, 'o--', color='0.5', label='shifted-band control')
ax.axvline(GMM_K, color='C3', ls=':', label=f'K inspected below = {GMM_K}')
ax.set(xlabel='Number of clusters K', ylabel='BIC (lower is preferred)', title='Does BIC support a particular K?')
ax.legend(frameon=False)
plt.show()

gmm_result = fit_gmm_motifs(env_1hz, GMM_K, seed=0, n_init=5)
stability_scores, mean_stability = seed_stability(env_1hz, GMM_K, seeds=range(5))
print(f'Median assignment confidence: {np.median(gmm_result.confidence):.1%}')
print(f'Mean pairwise ARI across five seeds: {mean_stability:.3f}')
print('All pairwise seed ARIs:', np.round(stability_scores, 3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]})
for cluster, center in enumerate(gmm_result.centers):
    axes[0].plot(band_labels, center, 'o-', label=f'GMM motif {cluster}')
axes[0].axhline(0, color='0.7', lw=0.8)
axes[0].set(ylabel='Mean band envelope (z)', title='GMM spectral centers')
axes[0].tick_params(axis='x', rotation=35)
axes[0].legend(ncol=2, fontsize=8)
axes[1].imshow(gmm_result.labels[None, :], aspect='auto', interpolation='none',
               extent=[time_1hz[0] - ts[0], time_1hz[-1] - ts[0], 0, 1], cmap='tab20')
axes[1].set(yticks=[], ylabel='GMM label', xlabel='Time in window (s)', title='Which motif is assigned each second?')
plt.tight_layout()
plt.show()

### Do not over-interpret a colorful GMM timeline

A good candidate result has interpretable cluster centers, high assignment confidence, high seed stability, and evidence that real simultaneous bands behave differently from shifted bands. It should also generalize to held-out time periods. A colorful strip alone is not validation because the model must assign every second somewhere.

## 9. Is the 7–8 Hz pattern simply movement?

Running can produce real theta activity and can also introduce movement artifacts. Either way, a motif strongly tied to running should be described as a behavior-associated motif rather than a behavior-independent awake state.

We interpolate running speed onto the same one-second timestamps as the spectral envelopes, then compare it with the 4–8 Hz envelope and GMM labels. Correlation does not prove causation, but it exposes a major confound.

In [ ]:
theta_band = int(np.argmin(np.abs(DEFAULT_FREQ_BANDS.mean(axis=1) - 7)))
speed_1hz = np.interp(time_1hz, window['running_ts'], window['running'])
theta_1hz = env_1hz[:, theta_band]
theta_speed_rho = pd.Series(theta_1hz).corr(pd.Series(speed_1hz), method='spearman')
if not np.isfinite(theta_speed_rho):
    print('Theta–speed correlation is undefined because one series is constant or missing in this window.')

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True, gridspec_kw={'height_ratios': [1, 2, 2]})
axes[0].imshow(gmm_result.labels[None, :], aspect='auto', interpolation='none',
               extent=[time_1hz[0] - ts[0], time_1hz[-1] - ts[0], 0, 1], cmap='tab20')
axes[0].set(yticks=[], ylabel='GMM')
axes[1].plot(time_1hz - ts[0], theta_1hz, color='#6a3d9a', lw=0.8)
axes[1].set_ylabel('4–8 Hz envelope (z)')
axes[2].plot(time_1hz - ts[0], speed_1hz, color='#1b9e77', lw=0.8)
axes[2].set(ylabel='Running speed', xlabel='Time in window (s)')
fig.suptitle(f'Movement check: Spearman rho(theta, speed) = {theta_speed_rho:.3f}')
plt.tight_layout()
plt.show()

pd.DataFrame({
    'seconds': pd.Series(gmm_result.labels).value_counts().sort_index(),
    'median_speed': pd.Series(speed_1hz).groupby(gmm_result.labels).median(),
    'median_theta_z': pd.Series(theta_1hz).groupby(gmm_result.labels).median(),
})

## 10. Optional: run every valid EEG channel

The automated audit repeats the original/CAR and binary/continuous comparison for every valid channel. It saves:

- `01_raw_vs_car.png`
- `02_channel_eda.png`
- `03_representation_steps.png`
- `04_theta_vs_running.png`
- `05_motif_counts_all_channels.png`
- `channel_summary.csv`
- `summary.json`

This can take a long time because motif detection is repeated for every channel and reference condition. Leave `RUN_FULL_AUDIT=False` until the earlier plots look sensible.

In [ ]:
if RUN_FULL_AUDIT:
    full_summary = run_diagnostics(
        INPUT_PATH, OUTPUT_DIR, n_null=N_NULL_DEMO, seed=0
    )
    display(full_summary.head())
    print('Saved outputs to:', OUTPUT_DIR)
else:
    print('Full audit skipped. Set RUN_FULL_AUDIT=True in the setup cell when ready.')

## 11. Optional isoflurane transition check

To ask whether motifs change as anesthesia deepens, set `INPUT_PATH` to an extracted **isoflurane** window and set `RUN_TRANSITION_CHECK=True`. The code divides usable time into early and late thirds, leaving the middle third out to make the contrast clearer.

This is a first diagnostic, not the final transition analysis. The better final version should align time to measured isoflurane level and compare matched-duration windows across mice.

In [ ]:
if RUN_TRANSITION_CHECK:
    n = len(env_clean)
    early_env = env_clean[:n // 3]
    late_env = env_clean[-n // 3:]
    transition_rows = []
    for period, values in [('early iso', early_env), ('late iso', late_env)]:
        for representation in ['binary', 'continuous']:
            result = get_motifs(
                [values.T], fs=fs, n_null=N_NULL_DEMO, seed=0, representation=representation
            )
            transition_rows.append({
                'period': period,
                'representation': representation,
                'motif_pairs': result.n_motifs,
            })
    transition_summary = pd.DataFrame(transition_rows)
    display(transition_summary)
else:
    print('Transition check skipped. Use an isoflurane input and enable the switch when ready.')

## 12. How to decide why an EEG channel failed

Use the following logic for each successful and unsuccessful channel:

| Observation | Likely interpretation |
|---|---|
| Continuous method succeeds but binary fails | Binarization probably removed useful magnitude information |
| CAR succeeds but original reference fails | Shared activity/noise was masking channel-specific structure |
| Original succeeds but CAR fails | The motif may be spatially widespread, or CAR removed real shared signal |
| High 60 Hz fraction and unstable motifs | Electrical noise is a likely problem |
| Strong 7–8 Hz activity tracks speed | The pattern is behavior-associated or movement-contaminated |
| Similar channels in one region succeed together | Anatomical location may explain motif recovery |
| GMM changes drastically across seeds | The apparent clusters are not reliable |
| Low-K GMM mainly separates awake from iso | The model found broad brain state, not detailed within-awake motifs |
| Early and late iso differ | Treating all isoflurane time as one stationary state may hide transitions |

The final deliverable should show representative successful and failed channels, then summarize all channels quantitatively. Avoid choosing only the prettiest example.

## Final-analysis checklist

Before treating a result as evidence:

- [ ] Use enough null permutations for the hierarchical method.
- [ ] Run at least five GMM seeds and report stability.
- [ ] Compare multiple K values instead of selecting the prettiest plot.
- [ ] Compare original reference with CAR.
- [ ] Include every valid EEG channel and its anatomical location.
- [ ] Overlay running, stimulation, and available pupil data.
- [ ] Separate awake, induction, early iso, and late iso appropriately.
- [ ] Test on held-out time windows or a second mouse.
- [ ] Save parameter values, tables, and plots needed to reproduce the conclusion.